In [3]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import load_model, Sequential
from tensorflow.keras.utils import to_categorical, plot_model

import matplotlib.pyplot as plt
import numpy as np

import warnings
from warnings import filterwarnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
filterwarnings('ignore')

In [4]:
#mnist veri setinin yüklenmesi
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Encoding
encoding işlemini bağımlı değişkene/labela/outputa uygulayacağız

*   Önce:[0,1,2,3,4,5,6,7,8,9]
*   Sonra:[0,1,0,0,0,0,0,0,0,0]

burada labelımız 2 sınıfına ait. yani görselde 2 var. çünkü encode ettiğimiz arrayde yalnızca 2. index 1 diğerleri 0.



In [5]:
y_train[0:5]   #verisetindeki 0dan 5e kadar olan verilerin etiketleri

array([5, 0, 4, 1, 9], dtype=uint8)

In [6]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

#verilerin ifade ediliş tarzı one-hot encoding ile değiştirilmiş oldu!

In [7]:
y_train[0:5]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

Bu satırlar, makine öğrenmesi (özellikle sinir ağları) için çok önemli bir kavramı gösteriyor: **one-hot encoding**.
---

### 🎯 Önce: Ne yapıyor bu kod?

```python
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
```

Bu satırlar, etiketleri (`y_train`, `y_test`) **one-hot encoding** denilen biçime dönüştürüyor.

---

### 📦 Örnekle açıklayalım:

MNIST veri setinde her örnek bir **rakamı (0–9)** temsil eder.
Yani normalde `y_train` şöyle görünür:

```
[5, 0, 4, 1, 9, ...]
```

Bunlar sadece **sınıf numaralarıdır** (yani etiketler).

Ama sinir ağları bu haliyle sınıf numaralarını anlamaz; onlar **sayısal vektörleri** sever 🙂
O yüzden her rakamı 10 elemanlı bir vektöre dönüştürürüz:

| Rakam | One-hot karşılığı     |
| ----- | --------------------- |
| 0     | [1,0,0,0,0,0,0,0,0,0] |
| 1     | [0,1,0,0,0,0,0,0,0,0] |
| 2     | [0,0,1,0,0,0,0,0,0,0] |
| 5     | [0,0,0,0,0,1,0,0,0,0] |
| 9     | [0,0,0,0,0,0,0,0,0,1] |

Yani, her etiket artık 10 boyutlu bir **sınıf olasılığı vektörü** hâline gelir.

---

### 🧠 Peki neden gerekli?

Çünkü:

1. **Sinir ağının çıkış katmanı** genelde `softmax` fonksiyonuyla biter.

   * Bu fonksiyon 10 sınıf için **olasılık dağılımı** üretir (örneğin `[0.02, 0.01, 0.9, ...]` gibi).
2. **Kayıp fonksiyonu (loss function)** genellikle `categorical_crossentropy` olur.

   * Bu fonksiyon, modelin çıktısını **one-hot etiketlerle** karşılaştırarak hatayı hesaplar.

Eğer etiketler sadece `[5, 0, 4, ...]` şeklinde olsaydı, model bu hatayı doğru şekilde hesaplayamazdı.

---

### 🔍 Kısaca özet:

| Neden | Açıklama                                                                           |
| ----- | ---------------------------------------------------------------------------------- |
| 🔢    | Sınıf etiketleri kategoriktir (sayısal değil).                                     |
| 🧮    | Sinir ağı her sınıf için olasılık tahmin eder.                                     |
| ⚙️    | `softmax` + `categorical_crossentropy` birlikte çalışabilsin diye one-hot gerekir. |
| 🎯    | Model her sınıf için olasılık öğrenir, sadece sayı değil.                          |

---

### 🌟 Sonuç:

One-hot encoding, etiketleri sinir ağının anlayabileceği biçime dönüştürür.
Yani model “bu görüntü **5**’tir” demek yerine “bu görüntü **%95 olasılıkla 5**, %3 olasılıkla 3, %2 olasılıkla 8” diyebilir.

Bu da **çok sınıflı sınıflandırmanın temelidir.** 🚀


## Reshaping
28X28 60000 tane train verimiz var ancak bunları pixel olarak da ifade etmemiz gerekiyor.

In [13]:
x_train.shape

(60000, 28, 28)

In [15]:
x_train.shape[0]

60000

In [14]:
image_size = x_train.shape[1]

In [17]:
image_size

28

In [18]:
x_train = x_train.reshape(x_train.shape[0], image_size, image_size, 1)
x_test = x_test.reshape(x_test.shape[0], image_size, image_size, 1)

print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")

x_train shape: (60000, 28, 28, 1)
x_test shape: (10000, 28, 28, 1)


Bu kısım da **veri ön işleme** aşamasının çok kritik bir adımı — özellikle **Convolutional Neural Network (CNN)**’lerle çalışırken 💪
Gel birlikte adım adım inceleyelim:

---

### 🔍 Kod ne yapıyor?

```python
x_train = x_train.reshape(x_train.shape[0], image_size, image_size, 1)
x_test = x_test.reshape(x_test.shape[0], image_size, image_size, 1)
```

Bu satırlar, **giriş verisinin (görüntülerin)** şeklini (boyutunu) CNN’in anlayacağı formata dönüştürüyor.

---

### 🧩 Adım adım açıklama

#### 🖼️ 1. MNIST görüntülerinin orijinal hali:

MNIST veri setindeki her görüntü 28x28 boyutunda **gri tonlamalı** bir resimdir.

Yani her örnek (örneğin bir “5” rakamı) şu şekilde temsil edilir:

```
(28, 28)
```

Bu, 2 boyutlu bir matristir (satır, sütun).

---

#### 🔁 2. CNN’ler 4 boyutlu giriş ister

Keras veya TensorFlow’daki CNN katmanları (`Conv2D`) giriş verisini şu **4 boyutlu** yapıda ister:

```
(samples, height, width, channels)
```

Bu parametreler:

* **samples** → veri sayısı (kaç tane resim var)
* **height** → resmin yüksekliği (örneğin 28)
* **width** → resmin genişliği (örneğin 28)
* **channels** → renk kanalı sayısı

  * 1 → siyah-beyaz (grayscale)
  * 3 → RGB (renkli)

---

#### 🔧 3. Kodun yaptığı şey:

```python
x_train.reshape(x_train.shape[0], image_size, image_size, 1)
```

* `x_train.shape[0]` → örnek sayısı (örneğin 60,000)
* `image_size` → 28 (yükseklik ve genişlik)
* `1` → **kanal sayısı** (gri tonlu olduğu için tek kanal)

Yani her görüntü artık şu boyutta:

```
(28, 28, 1)
```

Ve tüm eğitim setinin boyutu:

```
x_train shape: (60000, 28, 28, 1)
x_test shape: (10000, 28, 28, 1)
```

---

### 🧠 Neden bu gerekli?

Çünkü CNN katmanları **3 boyutlu giriş** (yükseklik, genişlik, kanal) bekler.
Eğer bu “kanal boyutu” eklenmezse, model giriş verisini 2D zannedip hata verir.

---

### 🎯 Kısaca özet:

| Ne                 | Anlamı                          | Örnek                     |
| ------------------ | ------------------------------- | ------------------------- |
| `x_train.shape[0]` | Görüntü sayısı                  | 60000                     |
| `image_size`       | Görüntü yüksekliği ve genişliği | 28                        |
| `1`                | Renk kanalı (grayscale)         | Tek kanal                 |
| Sonuç boyut        | (örnek_sayısı, 28, 28, 1)       | CNN girişine uygun format |

---

💬 **Kısaca:**
Bu kod, “her görüntüye bir kanal boyutu ekleyerek CNN’in anlayacağı 4D forma getiriyor.”
Bir bakıma modeli eğitmeden önce veriyi “konuşabileceği dile çeviriyor” diyebiliriz 🧩✨


## Normalization
her pixel üzerinde 0-255 arasında değerler vardo. bunlara 0-1 arasında olsun diyerek standartlaştıracağız.
peki neden?
eğitim süresinin daha hızlı olmasını sağlamaktır amaç.

In [19]:
x_train = x_train.astype('float32')/255
x_test = x_test.astype('float32')/255

x_train = x_train / 255.0
x_test = x_test / 255.0
🧠 Peki neden bunu yapıyoruz?
1. ⚙️ Modelin öğrenmesini kolaylaştırmak için
Sinir ağları sayısal ölçeklere çok duyarlıdır.
Piksel değerleri 0–255 aralığında olursa,

ağırlıklar çok büyük değerlerle çarpılır,

gradyanlar (hatalar) çok büyük veya çok küçük olabilir.

Bu da öğrenmeyi zorlaştırır veya kararsız hale getirir.

Ama veriyi 0–1 arasına sıkıştırırsan:

değerler dengeli olur,

ağ daha hızlı ve kararlı öğrenir,

gradyanlar taşmaz (overflow) veya sıfırlanmaz (vanish).

2. ⚡ Eğitim süresini kısaltmak için
Normalizasyon yapılmazsa optimizasyon algoritması (örneğin Adam, SGD)
her feature’ın (pikselin) farklı ölçekte olmasından dolayı zigzag şekilde ilerler.
Normalize edilirse daha doğrudan ve hızlı yakınsama olur.

Yani model daha az epoch’ta daha iyi sonuç verir. ⏱️

3. 📉 Ağırlık güncellemelerinin tutarlı olması için
CNN katmanlarındaki ağırlıklar genelde küçük rastgele sayılarla başlar (örneğin -0.1 ile 0.1 arası).
Eğer girdiler çok büyük olursa (örneğin 255), bu ağırlıklar aniden çok büyük güncellemeler alabilir — model “şoka girer”.
0–1 aralığında veriler bu etkiyi azaltır.

🔍 4. Matematiksel olarak daha stabil işlemler
Birçok aktivasyon fonksiyonu (sigmoid, tanh, ReLU) küçük sayılarla çalışırken daha düzgün gradyan üretir.
Örneğin sigmoid(255) ≈ 1 → gradyan ≈ 0 (öğrenme durur).
Ama sigmoid(0.8) → gayet iyi öğrenme sağlar.

🧩 Kısaca özet:
Sebep	Açıklama
⚙️ Öğrenmeyi kararlı hale getirir	Gradyan patlamasını / kaybolmasını önler
⚡ Eğitim süresini hızlandırır	Optimizasyonu kolaylaştırır
📉 Ağırlıkları dengeler	Her pikselin etkisi eşit olur
🔬 Aktivasyon fonksiyonları daha verimli çalışır	Özellikle sigmoid / tanh için önemli

💬 Özetle:
Normalizasyon, veriyi “modelin anlayacağı ölçeğe” indirger.
Böylece model, daha hızlı, daha kararlı ve daha doğru öğrenir. 🚀